In [43]:
import pandas as pd
import ast
import os
from collections import Counter

BASE = r"C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework"

# ── 1. 두 파일 로드 및 병합 ──────────────────────────────────────
df_old = pd.read_csv(BASE + r"\data\processed\knewnew_without_ad_with_keywords_04-12.csv")
df_new = pd.read_csv(BASE + r"\data\processed\knewnew_1-01_04-22_keywords_checkpoint.csv")

df_old = df_old.drop(columns=["광고여부"], errors="ignore")

df_merged = pd.concat([df_old, df_new], ignore_index=True).reset_index(drop=True)

print(f"df_old : {len(df_old):,}행")
print(f"df_new : {len(df_new):,}행")
print(f"병합 후: {len(df_merged):,}행")

# ── 2. 키워드 정규화 (공백 제거) ─────────────────────────────────
def remove_spaces(raw):
    if pd.isna(raw):
        return raw
    words = [w.replace(" ", "") for w in str(raw).split(",") if w.strip()]
    return ", ".join(words)

df_merged["keywords_v2"] = df_merged["keywords"].apply(remove_spaces)

# ── 3. 저장 ───────────────────────────────────────────────────────
output_file = BASE + r"\data\processed\knewnew_full_2025_v2.csv"
df_merged.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {output_file}")
print(df_merged[["url", "keywords_v2"]].head(3))


df_old : 1,037행
df_new : 471행
병합 후: 1,508행

저장 완료: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\knewnew_full_2025_v2.csv
                                        url  \
0  https://www.instagram.com/p/DIs30jKzrXh/   
1  https://www.instagram.com/p/DIsOpVSzL0D/   
2  https://www.instagram.com/p/DIsH68vTAo1/   

                                         keywords_v2  
0  럭셔리, 스테이, 리조트, 아만다리, 호시노야가루이자와, 스테이폴리오, 인플루언서,...  
1    롯데리아, 나폴리, 감자연구소, 디저트, 쥐포튀김, 떼리앙, 캐릭터, 띠부씰, 콜라보  
2            국밥, 뉴웨이브, MZ세대, 힙함, 감성, 이색식재료, 플레이팅, 한끼  


In [44]:
# ── 4. 키워드 빈도 카운트 (오름차순) ─────────────────────────────
def safe_parse(kw_str):
    if pd.isna(kw_str):
        return []
    try:
        result = ast.literal_eval(str(kw_str))
        if isinstance(result, list):
            return [k.strip() for k in result if k.strip()]
    except Exception:
        pass
    return [k.strip() for k in str(kw_str).split(",") if k.strip()]

df_merged["kw_list"] = df_merged["keywords_v2"].apply(safe_parse)

all_keywords = [kw for kws in df_merged["kw_list"] for kw in kws]
kw_freq = pd.DataFrame(Counter(all_keywords).items(), columns=["keyword", "count"])
kw_freq = kw_freq.sort_values("count", ascending=True).reset_index(drop=True)

print(f"전체 고유 키워드 수: {len(kw_freq):,}개")
kw_freq  # 빈도 낮은 순으로 전체 확인


전체 고유 키워드 수: 5,120개


,keyword,count
0,BCD,1
1,북창동,1
2,꾸미기,1
3,초코펜,1
4,글레이즈드,1
...,...,...
5115,커피,134
5116,맛집,137
5117,F&B,166
5118,디저트,233


In [45]:
# ── 고빈도 키워드 확인 (불용어 선별용) ──────────────────────────
# Cell 1 실행 후 사용 (kw_freq 필요)

# 빈도 내림차순 상위 100개 출력
top100 = kw_freq.sort_values("count", ascending=False).head(100)
print(f"상위 100개 키워드 (전체 {len(kw_freq):,}개 중):")
top100


상위 100개 키워드 (전체 5,120개 중):


,keyword,count
5119,카페,357
5118,디저트,233
5117,F&B,166
5116,맛집,137
5115,커피,134
...,...,...
5013,한정,15
5019,성수동,15
5022,전주,15
5025,퓨전,15


In [46]:
# ── 영어 키워드만 추출 (오름차순) ────────────────────────────────
import re

def is_english(kw):
    # 알파벳(+숫자/공백/특수문자) 포함, 한글 미포함
    return bool(re.search(r"[a-zA-Z]", kw)) and not bool(re.search(r"[가-힣]", kw))

kw_freq_eng = kw_freq[kw_freq["keyword"].apply(is_english)].reset_index(drop=True)

print(f"영어 키워드 수: {len(kw_freq_eng):,}개 / 전체 {len(kw_freq):,}개")
kw_freq_eng  # 빈도 낮은 순 전체 확인


영어 키워드 수: 114개 / 전체 5,120개


,keyword,count
0,BCD,1
1,I/Ocafe,1
2,RuineCoffee,1
3,Delicatessenplace,1
4,CentreCulturelChambreNoire,1
...,...,...
109,LP,13
110,Knewnew,17
111,GS25,25
112,CU,36


In [47]:
# ── 빈도 & Degree 분포 탐색 (필터링 기준 결정용) ─────────────────
# Cell 0, Cell 1 실행 후 사용 (df_merged, kw_list 필요)
import networkx as nx
from itertools import combinations

# 빠른 네트워크 구축
_G_tmp = nx.Graph()
_kw_doc_freq = {}

for kws in df_merged["kw_list"]:
    for kw in set(kws):
        _kw_doc_freq[kw] = _kw_doc_freq.get(kw, 0) + 1
    if len(kws) >= 2:
        for u, v in combinations(kws, 2):
            if _G_tmp.has_edge(u, v):
                _G_tmp[u][v]["weight"] += 1
            else:
                _G_tmp.add_edge(u, v, weight=1)

# freq + degree 테이블
rows = []
for node in _G_tmp.nodes():
    rows.append({
        "keyword": node,
        "freq": _kw_doc_freq.get(node, 0),
        "degree": _G_tmp.degree(node)
    })

fd_df = pd.DataFrame(rows).sort_values(["freq", "degree"], ascending=[True, True]).reset_index(drop=True)

print(f"전체 노드 수: {len(fd_df):,}개")
print(f"freq=1 AND degree=1: {len(fd_df[(fd_df.freq==1) & (fd_df.degree==1)]):,}개")
print(f"freq<=2 AND degree<=2: {len(fd_df[(fd_df.freq<=2) & (fd_df.degree<=2)]):,}개")
print(f"freq<=3 AND degree<=3: {len(fd_df[(fd_df.freq<=3) & (fd_df.degree<=3)]):,}개")
fd_df  # 오름차순으로 전체 확인 — 여기서 기준 결정


전체 노드 수: 5,120개
freq=1 AND degree=1: 0개
freq<=2 AND degree<=2: 0개
freq<=3 AND degree<=3: 0개


,keyword,freq,degree
0,음색,1,4
1,목소리,1,4
2,유학생,1,4
3,상륙,1,4
4,연세유업,1,4
...,...,...,...
5115,커피,134,553
5116,맛집,137,595
5117,F&B,166,705
5118,디저트,233,967


In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import os

try:
    import community as community_louvain
except ImportError:
    os.system("pip install python-louvain")
    import community as community_louvain

try:
    from pyvis.network import Network
except ImportError:
    os.system("pip install pyvis")
    from pyvis.network import Network


BASE = r"C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework"

MEGATREND_NAMES = {
    1:  "감성 카페 라이프",
    6:  "시즌 한정 디저트",
    7:  "핫플레이스 다이닝",
    8:  "선물형 디저트 브랜딩",
    10: "제철 과일 디저트",
    2:  "팝업·콜라보 경험",
    4:  "레트로 로컬 푸드",
    22: "혼술·홈술 문화",
    12: "가성비 야식·노포",
    18: "라이프스타일 쇼핑",
    17: "일식 열풍",
    3:  "빵지순례 문화",
    11: "추억·향수 간식",
    20: "건강식·비건",
    19: "미디어 푸드 콘텐츠",
    5:  "미식 체험",
    0:  "먹방·여행 콘텐츠",
    9:  "골목 술자리 문화",
    23: "수험생·이벤트 특수",
    13: "비주얼 데이트 플레이스",
    14: "프리미엄 파인다이닝",
    15: "도심 SNS 스팟",
    30: "연말연시 이벤트",
    21: "프리미엄 버거 재료",
    28: "시즌 한정 음료",
    27: "럭셔리 브랜드 F&B",
    24: "패션·다이닝 크로스오버",
    26: "로컬 여행 맛집",
    16: "빈티지 라이프스타일",
    31: "인플루언서 맛집",
    25: "간편결제 라이프",
    29: "대학가 음식 문화",
    32: "엔터테인먼트 스팟",
    33: "배달·푸드테크",
}

PALETTE = [
    "#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6",
    "#1ABC9C", "#E67E22", "#34495E", "#E91E63", "#00BCD4",
    "#8BC34A", "#FF5722", "#607D8B", "#795548", "#FF9800",
    "#673AB7", "#009688", "#F44336", "#2196F3", "#4CAF50",
    "#CDDC39", "#FF5252", "#40C4FF", "#69F0AE", "#FFD740",
    "#E040FB", "#00E5FF", "#76FF03", "#FF6D00", "#651FFF",
    "#F50057", "#00B0FF", "#C6FF00", "#FFAB40", "#D500F9",
]


def safe_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        if isinstance(x, str):
            cleaned = x.strip("[]")
            return [kw.strip() for kw in cleaned.split(",") if kw.strip()]
        return []


def build_network(file_path, column_name, freq_threshold=2, degree_threshold=2):
    print("[Step 1] 네트워크 생성 중...")
    df = pd.read_csv(file_path)
    df[column_name] = df[column_name].apply(safe_eval)

    total_docs = len(df)
    kw_doc_freq = Counter()
    for keywords in df[column_name]:
        for kw in set(keywords):
            kw_doc_freq[kw] += 1

    high_freq_stopwords = {kw for kw, freq in kw_doc_freq.items() if freq >= total_docs * 0.30}
    manual_stopwords = {
        "뉴뉴", "뉴뉴매거진", "Knewnew", "Knewnewmagazine",
    }
    stopwords = high_freq_stopwords | manual_stopwords
    print(f"  불용어 제거: 총 {len(stopwords)}개")

    G = nx.Graph()
    for keywords in df[column_name]:
        if not isinstance(keywords, list) or len(keywords) < 2:
            continue
        filtered = [kw for kw in keywords if kw not in stopwords]
        if len(filtered) < 2:
            continue
        for u, v in combinations(filtered, 2):
            if G.has_edge(u, v):
                G[u][v]["weight"] += 1
            else:
                G.add_edge(u, v, weight=1)

    print(f"  노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")

    # 저빈도 + 저degree 노드 동시 제거
    nodes_to_remove = [
        node for node in G.nodes()
        if kw_doc_freq.get(node, 0) < freq_threshold and G.degree(node) < degree_threshold
    ]
    G.remove_nodes_from(nodes_to_remove)
    print(f"  노이즈 노드 제거: {len(nodes_to_remove)}개 (freq<{freq_threshold} AND degree<{degree_threshold})")
    print(f"  필터링 후 — 노드: {G.number_of_nodes()}개 | 엣지: {G.number_of_edges()}개")

    return G, kw_doc_freq


def detect_communities(G):
    print("\n[Step 2] Louvain 커뮤니티 탐지 중...")
    partition = community_louvain.best_partition(G, weight="weight", random_state=42)

    community_sizes = Counter(partition.values())
    print(f"  발견된 군집 수: {len(community_sizes)}개")

    deg_cent = nx.degree_centrality(G)
    print("\n[군집별 Top 5 키워드 (연결 중심성 기준)]")
    print("-" * 55)
    for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
        members = [node for node, c in partition.items() if c == comm_id]
        top5 = sorted(members, key=lambda n: deg_cent[n], reverse=True)[:5]
        print(f"  군집 {comm_id:2d} ({len(members):4d}개): {top5}")
    print("-" * 55)

    return partition


def build_visualization(G, partition, save_path="trend_network_full.html"):
    print("\n[Step 3] 시각화 HTML 생성 중...")

    deg_cent = nx.degree_centrality(G)

    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality(G, max_iter=1000)

    eig_values = list(eig_cent.values())
    eig_min, eig_max = min(eig_values), max(eig_values)

    def eig_to_border(val):
        normalized = (val - eig_min) / (eig_max - eig_min + 1e-9)
        return 1 + normalized * 9

    community_sizes = Counter(partition.values())
    comm_rank = {
        comm_id: rank
        for rank, (comm_id, _) in enumerate(
            sorted(community_sizes.items(), key=lambda x: x[1], reverse=True)
        )
    }

    net = Network(height="950px", width="100%", bgcolor="#1a1a2e", font_color="#ffffff", notebook=False)
    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.005,
          "springLength": 100,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "maxVelocity": 50,
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"enabled": true, "iterations": 300, "updateInterval": 50, "fit": true}
      },
      "nodes": {
        "shape": "dot",
        "font": {"size": 12, "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"},
        "borderWidthSelected": 4
      },
      "edges": {"smooth": {"enabled": true, "type": "continuous"}},
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "hideEdgesOnDrag": true,
        "navigationButtons": true,
        "keyboard": true,
        "selectConnectedEdges": true
      }
    }
    """)

    for node in G.nodes():
        comm_id      = partition[node]
        megatrend    = MEGATREND_NAMES.get(comm_id, f"군집 {comm_id}")
        color        = PALETTE[comm_rank.get(comm_id, 0) % len(PALETTE)]
        size         = min(5 + deg_cent[node] * 120, 40)
        border_width = eig_to_border(eig_cent[node])
        net.add_node(
            node,
            label=node,
            color={
                "background": color,
                "border":     "#ffffff",
                "highlight":  {"background": color, "border": "#FFD700"},
                "hover":      {"background": color, "border": "#FFD700"},
            },
            size=size,
            borderWidth=border_width,
            title=(
                f"키워드: {node}\n"
                f"메가트렌드: {megatrend}\n"
                f"─────────────────\n"
                f"크기   → 대세  (연결 중심성):     {deg_cent[node]:.4f}\n"
                f"테두리 → 파워  (고유벡터 중심성): {eig_cent[node]:.4f}\n"
                f"─────────────────\n"
                f"군집 크기: {community_sizes[comm_id]}개"
            ),
            group=str(comm_id)
        )

    max_weight = max(d["weight"] for _, _, d in G.edges(data=True))
    for u, v, data in G.edges(data=True):
        w = data["weight"]
        net.add_edge(
            u, v,
            value=w,
            width=0.5 + 2.5 * (w / max_weight),
            color={"opacity": 0.05 + 0.4 * (w / max_weight)},
            title=f"{u} ↔ {v}\n동시 출현: {w}회"
        )

    net.save_graph(save_path)

    # 클릭 인터랙션 JS 주입
    with open(save_path, "r", encoding="utf-8") as f:
        html = f.read()

    custom_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {

  network.once("stabilized", function() {
    network.setOptions({ physics: { enabled: false } });
  });

  network.on("click", function(params) {
    if (params.nodes.length > 0) {
      var selectedNode   = params.nodes[0];
      var connectedNodes = network.getConnectedNodes(selectedNode);
      var connectedEdges = network.getConnectedEdges(selectedNode);

      nodes.update(nodes.get().map(function(n) {
        return { id: n.id, opacity: (n.id === selectedNode || connectedNodes.indexOf(n.id) !== -1) ? 1.0 : 0.05 };
      }));
      edges.update(edges.get().map(function(e) {
        return { id: e.id, color: { opacity: connectedEdges.indexOf(e.id) !== -1 ? 0.9 : 0.02 } };
      }));

    } else {
      nodes.update(nodes.get().map(function(n) { return { id: n.id, opacity: 1.0 }; }));
      edges.update(edges.get().map(function(e) { return { id: e.id, color: { opacity: null } }; }));
    }
  });
});
</script>
"""
    html = html.replace("</body>", custom_js + "\n</body>")
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"  저장 완료 → '{save_path}'")
    print("  - 노드 클릭: 연결 노드만 강조 / 빈 곳 클릭: 전체 복원")
    print("  - 안정화 후 물리 엔진 자동 OFF")


# ── 실행 ─────────────────────────────────────────────────────────
FILE_PATH        = BASE + r"\data\processed\knewnew_full_2025_v2.csv"
COLUMN_NAME      = "keywords_v2"
SAVE_PATH        = "./trend_network_full.html"
FREQ_THRESHOLD   = 2
DEGREE_THRESHOLD = 2

G, kw_doc_freq = build_network(FILE_PATH, COLUMN_NAME, FREQ_THRESHOLD, DEGREE_THRESHOLD)
partition      = detect_communities(G)
build_visualization(G, partition, SAVE_PATH)


IndentationError: expected an indented block after 'try' statement on line 9 (58914050.py, line 10)

In [ ]:
# ── 이웃 Jaccard 유사도 기반 동의어 후보 탐색 ────────────────────
# Cell 9 실행 후 G가 있는 상태에서 실행

from itertools import combinations

TOP_N       = 50    # 상위 몇 쌍 출력
MIN_JACCARD = 0.3   # 이 이상만 후보로 간주

# 공통 이웃이 있는 쌍만 대상으로 Jaccard 계산
neighbor_sets = {node: set(G.neighbors(node)) for node in G.nodes()}

candidate_pairs = set()
for node in G.nodes():
    for neighbor in neighbor_sets[node]:
        for other in neighbor_sets[neighbor]:
            if other != node:
                pair = tuple(sorted([node, other]))
                candidate_pairs.add(pair)

print(f"공통 이웃 쌍 수: {len(candidate_pairs):,}개 탐색 중...")

rows = []
for u, v in candidate_pairs:
    n_u = neighbor_sets[u]
    n_v = neighbor_sets[v]
    intersection = len(n_u & n_v)
    union        = len(n_u | n_v)
    jaccard      = intersection / union if union > 0 else 0
    if jaccard >= MIN_JACCARD:
        rows.append({"keyword_A": u, "keyword_B": v, "jaccard": round(jaccard, 4),
                     "common_neighbors": intersection})

jaccard_df = pd.DataFrame(rows).sort_values("jaccard", ascending=False).reset_index(drop=True)

print(f"Jaccard >= {MIN_JACCARD} 후보 쌍: {len(jaccard_df)}개")
print(f"상위 {TOP_N}개:")
jaccard_df.head(TOP_N)


In [ ]:
# ── Jaccard 결과에서 영어↔한글 교차 쌍만 필터링 ─────────────────
# jaccard_df가 있는 상태에서 실행 (Cell 6 이후)
import re

def is_english(kw):
    return bool(re.search(r"[a-zA-Z]", kw)) and not bool(re.search(r"[가-힣]", kw))

def is_korean(kw):
    return bool(re.search(r"[가-힣]", kw))

cross_lang = jaccard_df[
    (jaccard_df["keyword_A"].apply(is_english) & jaccard_df["keyword_B"].apply(is_korean)) |
    (jaccard_df["keyword_A"].apply(is_korean)  & jaccard_df["keyword_B"].apply(is_english))
].reset_index(drop=True)

print(f"영어↔한글 교차 후보 쌍: {len(cross_lang)}개")
cross_lang


In [8]:
# 기존 코드 실행 후 G, partition이 있는 상태에서 아래 추가 실행

import pandas as pd
import networkx as nx
from collections import Counter

deg_cent = nx.degree_centrality(G)
community_sizes = Counter(partition.values())

rows = []
for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
    members = [node for node, c in partition.items() if c == comm_id]
    top10 = sorted(members, key=lambda n: deg_cent[n], reverse=True)[:10]
    rows.append({
        '군집ID':    comm_id,
        '키워드수':  community_sizes[comm_id],
        'Top10키워드': ', '.join(top10),
        '메가트렌드명': ''  # 나중에 직접 채울 칸
    })

result_df = pd.DataFrame(rows)
result_df.to_csv('./megatrend_naming.csv',
                 index=False, encoding='utf-8-sig')
print(result_df[['군집ID','키워드수','Top10키워드']].to_string())

    군집ID  키워드수                                                     Top10키워드
0      1   832                   카페, 디저트, 맛집, 커피, 브런치, 데이트, 여행, 빈티지, 감성, 힐링
1      6   451               말차, CU, 딸기, 제주, 스타벅스, 아이스크림, 피스타치오, 굿즈, 망고, 미국
2      7   394              F&B, 성수, 와인, 연말, 크리스마스, 샌드위치, 파스타, 이자카야, 신상, 피자
3      8   250                   케이크, 겨울, 음식, 가을, 선물, 브랜드, 취향, 친구, 무화과, 디자인
4     10   227                     여름, 제철, 달콤, 빙수, 상큼, 과일, 파르페, 음료, 성심당, 셀럽
5      2   168               팝업, 콜라보, 타코, 플리마켓, 홍콩, 캐릭터, 성수동, 막걸리, 큐레이션, 파리
6      4   161                  레트로, 저녁, 점심, 버거, 만두, 아메리칸, 국밥, 닭갈비, 다방, 사랑방
7     22   149                     안주, 한식, 퓨전, 맥주, 연희동, 음악, 혼술, 햄버거, 맛, 위스키
8     12   146                  떡볶이, 오뎅, 하이볼, 가성비, 야식, 대구, 야장, 시원함, 플라워, 노포
9     18   132                      쇼핑, 도쿄, 라멘, 휴식, 족발, 메뉴, 편집샵, 루틴, 전시, 스시
10    17   128                  일본, 바삭, 일식, 직장인, 키링, 사시미, 감칠맛, 오리지널, 식감, 카츠
11     3   118              빵, 베이글, 빵지순례, 빵순이, 초여름, 슈톨렌, 카라멜, 빵덕후, 호두과자, 천안
12    11   1

In [20]:
# ── 군집별 내부 중심성 분석 (키워드 수 내림차순) ─────────────────
# Cell 5 실행 후 G, partition이 있는 상태에서 실행
import pandas as pd
import networkx as nx
from collections import Counter

community_sizes = Counter(partition.values())
results = []

for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
    members = [node for node, c in partition.items() if c == comm_id]
    sub_G   = G.subgraph(members).copy()

    # 군집 내부 중심성 계산
    deg  = nx.degree_centrality(sub_G)
    try:
        eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        eig = {n: 0.0 for n in sub_G.nodes()}
    bet  = nx.betweenness_centrality(sub_G, weight="weight")

    # 지표별 Top 5
    top_deg = sorted(deg, key=deg.get, reverse=True)[:5]
    top_eig = sorted(eig, key=eig.get, reverse=True)[:5]
    top_bet = sorted(bet, key=bet.get, reverse=True)[:5]

    results.append({
        "군집ID":        comm_id,
        "메가트렌드":    MEGATREND_NAMES.get(comm_id, f"군집 {comm_id}"),
        "키워드수":      size,
        "대세(Degree)":  ", ".join(top_deg),
        "파워(Eigen)":   ", ".join(top_eig),
        "융합허브(Bet)": ", ".join(top_bet),
    })

comm_df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)
comm_df


NameError: name 'MEGATREND_NAMES' is not defined

In [ ]:
# ── 군집별 서브네트워크 시각화 ───────────────────────────────────
# Cell 5 실행 후 G, partition, MEGATREND_NAMES, PALETTE가 있는 상태에서 실행
import os
from pyvis.network import Network
import networkx as nx
from collections import Counter

SAVE_DIR = "./community_networks"
os.makedirs(SAVE_DIR, exist_ok=True)

community_sizes = Counter(partition.values())

for comm_id, size in sorted(community_sizes.items(), key=lambda x: x[1], reverse=True):
    megatrend = MEGATREND_NAMES.get(comm_id, f"군집_{comm_id}")
    members   = [node for node, c in partition.items() if c == comm_id]
    sub_G     = G.subgraph(members).copy()

    # 군집 내 중심성
    deg = nx.degree_centrality(sub_G)
    try:
        eig = nx.eigenvector_centrality(sub_G, max_iter=1000, weight="weight")
    except nx.PowerIterationFailedConvergence:
        eig = {n: 0.0 for n in sub_G.nodes()}

    eig_vals = list(eig.values())
    eig_min, eig_max = min(eig_vals), max(eig_vals)

    def eig_to_border(val):
        return 1 + ((val - eig_min) / (eig_max - eig_min + 1e-9)) * 9

    color = PALETTE[list(sorted(community_sizes, key=community_sizes.get, reverse=True)).index(comm_id) % len(PALETTE)]

    net = Network(height="800px", width="100%", bgcolor="#1a1a2e", font_color="#ffffff", notebook=False)
    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "centralGravity": 0.01,
          "springLength": 80,
          "springConstant": 0.1,
          "damping": 0.5,
          "avoidOverlap": 0.8
        },
        "maxVelocity": 50,
        "minVelocity": 0.5,
        "solver": "forceAtlas2Based",
        "stabilization": {"enabled": true, "iterations": 200, "fit": true}
      },
      "nodes": {
        "shape": "dot",
        "font": {"size": 13, "face": "Apple SD Gothic Neo, Malgun Gothic, sans-serif"},
        "borderWidthSelected": 4
      },
      "edges": {"smooth": {"enabled": true, "type": "continuous"}},
      "interaction": {
        "hover": true,
        "tooltipDelay": 100,
        "hideEdgesOnDrag": true,
        "navigationButtons": true,
        "keyboard": true
      }
    }
    """)

    for node in sub_G.nodes():
        size_node    = min(8 + deg[node] * 150, 50)
        border_width = eig_to_border(eig[node])
        net.add_node(
            node,
            label=node,
            color={
                "background": color,
                "border":     "#ffffff",
                "highlight":  {"background": color, "border": "#FFD700"},
                "hover":      {"background": color, "border": "#FFD700"},
            },
            size=size_node,
            borderWidth=border_width,
            title=(
                f"키워드: {node}\n"
                f"대세(Degree):  {deg[node]:.4f}\n"
                f"파워(Eigen):   {eig[node]:.4f}"
            )
        )

    if sub_G.number_of_edges() > 0:
        max_w = max(d["weight"] for _, _, d in sub_G.edges(data=True))
        for u, v, data in sub_G.edges(data=True):
            w = data["weight"]
            net.add_edge(
                u, v,
                value=w,
                width=0.5 + 3.0 * (w / max_w),
                color={"opacity": 0.1 + 0.7 * (w / max_w)},
                title=f"{u} ↔ {v}\n동시 출현: {w}회"
            )

    safe_name = megatrend.replace("/", "_").replace("·", "_")
    save_path = f"{SAVE_DIR}/comm_{comm_id:02d}_{safe_name}.html"
    net.save_graph(save_path)

    # 안정화 후 물리 OFF + 클릭 인터랙션
    with open(save_path, "r", encoding="utf-8") as f:
        html = f.read()

    custom_js = """
<script>
document.addEventListener("DOMContentLoaded", function() {
  network.once("stabilized", function() {
    network.setOptions({ physics: { enabled: false } });
  });
  network.on("click", function(params) {
    if (params.nodes.length > 0) {
      var sel   = params.nodes[0];
      var conn  = network.getConnectedNodes(sel);
      var edges_ = network.getConnectedEdges(sel);
      nodes.update(nodes.get().map(function(n) {
        return { id: n.id, opacity: (n.id === sel || conn.indexOf(n.id) !== -1) ? 1.0 : 0.05 };
      }));
      edges.update(edges.get().map(function(e) {
        return { id: e.id, color: { opacity: edges_.indexOf(e.id) !== -1 ? 0.9 : 0.02 } };
      }));
    } else {
      nodes.update(nodes.get().map(function(n) { return { id: n.id, opacity: 1.0 }; }));
      edges.update(edges.get().map(function(e) { return { id: e.id, color: { opacity: null } }; }));
    }
  });
});
</script>
"""
    html = html.replace("</body>", custom_js + "\n</body>")
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"  [{comm_id:2d}] {megatrend:20s} | {len(members):4d}개 노드 → {save_path}")

print(f"\n총 {len(community_sizes)}개 군집 저장 완료 → {SAVE_DIR}/")
